# Air travel carrier operations

Analysis of [zkskhurram/airline-ticket-prices-vs-oil-and-fuel-costs](https://www.kaggle.com/datasets/zkskhurram/airline-ticket-prices-vs-oil-and-fuel-costs): oil and jet fuel, airline financials, ticket prices, fuel surcharges, conflict events, and route-level cost impact.

- **Financials**: `revenue_usd_m`, `fuel_cost_usd_m`, `net_profit_usd_m`; **capacity proxies**: `passengers_carried_m`, `fleet_size`, ticket `load_factor_pct`.
- **Phases**: `conflict_phase` and highlighted date bands (COVID-19, Ukraine, US–Iran scenario window in the synthetic series) are **illustrative context**, not causal identification.
- **Per-carrier**: passenger and fare time series for top carriers; Brent vs carrier fuel spend; carrier-level fare vs fuel-intensity scatter.
- **Correlation heatmap**: single-hue scale (strength = |ρ|, label = signed *r*); **radar** charts compare carriers, `region`, `conflict_phase`, and `country` on min–max-scaled axes.
- **Plotly on Kaggle**: iframe renderer via `show_plotly` ([workaround](https://www.kaggle.com/code/stpeteishii/solution-to-a-plotly-graph-cannot-be-displayed)).


In [1]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
from IPython.display import display, HTML
from plotly.subplots import make_subplots
import plotly.colors as pc

PRISM = list(px.colors.qualitative.Prism)
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = PRISM
px.defaults.color_continuous_scale = [
    [i / max(len(PRISM) - 1, 1), c] for i, c in enumerate(PRISM)
]

PHASE_ORDER = [
    "Pre-Pandemic Baseline",
    "COVID-19 Collapse",
    "Recovery & Surge",
    "Ukraine War Shock",
    "Stabilisation",
    "Gaza-Israel Conflict",
    "Pre-Iran Escalation",
    "US-Iran War Conflict",
]
PHASE_COLORS = {k: PRISM[i % len(PRISM)] for i, k in enumerate(PHASE_ORDER)}


def prism_rgba(color: str, alpha: float) -> str:
    c = color.strip()
    lc = c.lower()
    if lc.startswith("rgb"):
        r, g, b = (int(x) for x in pc.unlabel_rgb(c))
    else:
        h = c if c.startswith("#") else f"#{c}"
        r, g, b = pc.hex_to_rgb(h)
    return f"rgba({r},{g},{b},{alpha})"


_IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle").exists()
if _IS_KAGGLE:
    pio.renderers.default = "iframe"


def show_plotly(fig):
    fig.update_layout(paper_bgcolor="white", plot_bgcolor="white", colorway=PRISM)
    if os.environ.get("PLOTLY_FORCE_HTML", "").lower() in ("1", "true", "yes"):
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
    elif _IS_KAGGLE:
        fig.show()
    else:
        display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))


In [2]:
CSV_NAMES = {
    "oil": "oil_jet_fuel_prices.csv",
    "tick": "airline_ticket_prices.csv",
    "fin": "airline_financial_impact.csv",
    "sc": "fuel_surcharges.csv",
    "ev": "conflict_oil_events.csv",
    "rt": "route_cost_impact.csv",
}


def resolve_data_dir() -> Path:
    explicit = Path(
        "/kaggle/input/datasets/zkskhurram/airline-ticket-prices-vs-oil-and-fuel-costs"
    )
    if explicit.exists():
        return explicit
    base = Path("/kaggle/input")
    if base.exists():
        for p in base.rglob("oil_jet_fuel_prices.csv"):
            return p.parent
    local = Path("data")
    assert (local / CSV_NAMES["oil"]).exists(), (
        f"Missing {CSV_NAMES['oil']} under {local.resolve()} — run `make download`"
    )
    return local


DATA_DIR = resolve_data_dir()
print(f"DATA_DIR = {DATA_DIR}")

oil = pd.read_csv(DATA_DIR / CSV_NAMES["oil"])
tick = pd.read_csv(DATA_DIR / CSV_NAMES["tick"])
fin = pd.read_csv(DATA_DIR / CSV_NAMES["fin"])
sc = pd.read_csv(DATA_DIR / CSV_NAMES["sc"])
ev = pd.read_csv(DATA_DIR / CSV_NAMES["ev"])
rt = pd.read_csv(DATA_DIR / CSV_NAMES["rt"])

for name, df in [
    ("oil", oil),
    ("tickets", tick),
    ("financials", fin),
    ("surcharges", sc),
    ("events", ev),
    ("routes", rt),
]:
    print(f"  {name:<14} {df.shape[0]:>7,} rows × {df.shape[1]:>2} cols")


DATA_DIR = data
  oil                 87 rows × 12 cols
  tickets         14,355 rows × 18 cols
  financials         725 rows × 20 cols
  surcharges      10,092 rows × 13 cols
  events              36 rows × 14 cols
  routes           3,132 rows × 23 cols


In [3]:
def parse_month(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s.astype(str) + "-01", errors="coerce")


for df in (oil, tick, fin, sc, rt):
    if "month" in df.columns:
        df["month_dt"] = parse_month(df["month"])

if "event_date" in ev.columns:
    ev["event_dt"] = pd.to_datetime(ev["event_date"], errors="coerce")

assert oil["month_dt"].notna().all()


## Oil and jet fuel (with phase bands)

Three stacked panels: Brent vs jet fuel (with crack spread band), YoY Brent change, OPEC production. Vertical bands: COVID-19 collapse, Ukraine war shock, and US–Iran conflict window (as in the reference kernel style).


In [4]:
oil_sorted = oil.sort_values("month_dt").reset_index(drop=True)
months = oil_sorted["month_dt"]
PHASE_VRECT = {
    "COVID-19 Collapse": ("2020-03-01", "2021-05-01", PHASE_COLORS["COVID-19 Collapse"]),
    "Ukraine War Shock": ("2022-03-01", "2022-12-01", PHASE_COLORS["Ukraine War Shock"]),
    "US-Iran War Conflict": ("2025-12-01", "2026-03-01", PHASE_COLORS["US-Iran War Conflict"]),
}

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        "Brent crude vs jet fuel (USD/bbl)",
        "Year-over-year Brent change (%)",
        "OPEC production (million bbl/day)",
    ),
)

fig.add_trace(
    go.Scatter(
        x=months,
        y=oil_sorted["brent_crude_usd_barrel"],
        name="Brent",
        line=dict(color=PRISM[0], width=2),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=months,
        y=oil_sorted["jet_fuel_usd_barrel"],
        name="Jet fuel",
        line=dict(color=PRISM[1], width=2),
        fill="tonexty",
        fillcolor=prism_rgba(PRISM[1], 0.2),
    ),
    row=1,
    col=1,
)

yoy = oil_sorted["yoy_brent_change_pct"].fillna(0)
colors_bar = [PRISM[2] if v >= 0 else PRISM[3] for v in yoy]
fig.add_trace(
    go.Bar(
        x=months,
        y=yoy,
        name="YoY Brent %",
        marker_color=colors_bar,
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=months,
        y=oil_sorted["opec_production_mbd"],
        name="OPEC mbd",
        line=dict(color=PRISM[4], width=2),
        fill="tozeroy",
        fillcolor=prism_rgba(PRISM[4], 0.25),
        showlegend=False,
    ),
    row=3,
    col=1,
)

for _label, (x0, x1, c) in PHASE_VRECT.items():
    for row in (1, 2, 3):
        fig.add_vrect(
            x0=x0,
            x1=x1,
            fillcolor=c,
            opacity=0.12,
            layer="below",
            line_width=0,
            row=row,
            col=1,
        )

fig.update_layout(
    height=900,
    title_text="Oil and jet fuel market overview",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
fig.update_yaxes(title_text="USD/bbl", row=1, col=1)
fig.update_yaxes(title_text="%", row=2, col=1)
fig.update_yaxes(title_text="mbd", row=3, col=1)
show_plotly(fig)


## Revenue, cost, profit, and capacity (financial panel)

Industry totals by month: sum of `revenue_usd_m`, `fuel_cost_usd_m`, `net_profit_usd_m`, and `passengers_carried_m` across carriers. Non-fuel operating costs are not reported separately; implied non-fuel gap can be seen as revenue minus fuel minus profit.


In [5]:
fin_m = (
    fin.groupby("month_dt", as_index=False)
    .agg(
        revenue_usd_m=("revenue_usd_m", "sum"),
        fuel_cost_usd_m=("fuel_cost_usd_m", "sum"),
        net_profit_usd_m=("net_profit_usd_m", "sum"),
        passengers_carried_m=("passengers_carried_m", "sum"),
        fleet_size=("fleet_size", "sum"),
    )
    .sort_values("month_dt")
)
fin_m["implied_other_costs_usd_m"] = (
    fin_m["revenue_usd_m"] - fin_m["fuel_cost_usd_m"] - fin_m["net_profit_usd_m"]
)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Revenue, fuel cost, net profit (industry sum, USD millions)", "Passengers carried (millions)"),
)
fig.add_trace(
    go.Scatter(
        x=fin_m["month_dt"],
        y=fin_m["revenue_usd_m"],
        name="Revenue",
        line=dict(color=PRISM[0], width=2),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=fin_m["month_dt"],
        y=fin_m["fuel_cost_usd_m"],
        name="Fuel cost",
        line=dict(color=PRISM[1], width=2),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=fin_m["month_dt"],
        y=fin_m["net_profit_usd_m"],
        name="Net profit",
        line=dict(color=PRISM[2], width=2),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=fin_m["month_dt"],
        y=fin_m["implied_other_costs_usd_m"],
        name="Implied other costs (rev − fuel − profit)",
        line=dict(color=PRISM[3], width=2, dash="dot"),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=fin_m["month_dt"],
        y=fin_m["passengers_carried_m"],
        name="Passengers",
        line=dict(color=PRISM[4], width=2),
        showlegend=True,
    ),
    row=2,
    col=1,
)
for _label, (x0, x1, c) in PHASE_VRECT.items():
    for row in (1, 2):
        fig.add_vrect(
            x0=x0, x1=x1, fillcolor=c, opacity=0.1, layer="below", line_width=0, row=row, col=1
        )
fig.update_layout(height=700, title_text="Industry financials and capacity proxy", hovermode="x unified")
show_plotly(fig)


## Per-carrier passengers, fares, and oil

Top carriers by cumulative `passengers_carried_m` in the financials table: passenger volumes over time, mean ticket fare from the ticket file, and Brent vs each carrier’s monthly fuel cost (USD millions). Oil prices are **global**; carrier fuel spend reflects scale and hedging mix in the synthetic panel.


In [ ]:
# Top carriers by cumulative passengers (financials)
_pass_sum = fin.groupby("airline", as_index=False)["passengers_carried_m"].sum()
_top_carriers = _pass_sum.nlargest(10, "passengers_carried_m")["airline"].tolist()
fin_top = fin[fin["airline"].isin(_top_carriers)].copy()

fig_pax = px.line(
    fin_top.sort_values(["month_dt", "airline"]),
    x="month_dt",
    y="passengers_carried_m",
    color="airline",
    color_discrete_sequence=PRISM,
    title="Passengers carried (millions) — top 10 carriers by cumulative passengers",
)
for _label, (x0, x1, c) in PHASE_VRECT.items():
    fig_pax.add_vrect(
        x0=x0, x1=x1, fillcolor=c, opacity=0.08, layer="below", line_width=0
    )
fig_pax.update_layout(hovermode="x unified", height=520)
show_plotly(fig_pax)

tick_top = tick[tick["airline"].isin(_top_carriers)].copy()
fare_m = (
    tick_top.groupby(["month_dt", "airline"], as_index=False)["total_fare_usd"]
    .mean()
    .sort_values(["month_dt", "airline"])
)
fig_fare = px.line(
    fare_m,
    x="month_dt",
    y="total_fare_usd",
    color="airline",
    color_discrete_sequence=PRISM,
    title="Mean total fare (USD) by carrier — same top 10 (ticket rows)",
)
for _label, (x0, x1, c) in PHASE_VRECT.items():
    fig_fare.add_vrect(
        x0=x0, x1=x1, fillcolor=c, opacity=0.08, layer="below", line_width=0
    )
fig_fare.update_layout(hovermode="x unified", height=520)
show_plotly(fig_fare)

oil_brent = oil_sorted[["month_dt", "brent_crude_usd_barrel"]]
fin_fuel_m = (
    fin_top.groupby(["month_dt", "airline"], as_index=False)
    .agg(fuel_cost_usd_m=("fuel_cost_usd_m", "sum"))
    .sort_values(["month_dt", "airline"])
)
fig_oil_fuel = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    subplot_titles=(
        "Brent crude (USD/bbl)",
        "Monthly fuel cost (USD m) — top carriers",
    ),
)
fig_oil_fuel.add_trace(
    go.Scatter(
        x=oil_brent["month_dt"],
        y=oil_brent["brent_crude_usd_barrel"],
        name="Brent",
        line=dict(color="#2a2a2a", width=2.5),
    ),
    row=1,
    col=1,
)
for al in _top_carriers[:8]:
    sub = fin_fuel_m[fin_fuel_m["airline"] == al]
    fig_oil_fuel.add_trace(
        go.Scatter(
            x=sub["month_dt"],
            y=sub["fuel_cost_usd_m"],
            name=al,
            mode="lines",
            line=dict(width=1.6),
        ),
        row=2,
        col=1,
    )
for _label, (x0, x1, c) in PHASE_VRECT.items():
    for row in (1, 2):
        fig_oil_fuel.add_vrect(
            x0=x0,
            x1=x1,
            fillcolor=c,
            opacity=0.08,
            layer="below",
            line_width=0,
            row=row,
            col=1,
        )
fig_oil_fuel.update_layout(
    height=800,
    hovermode="x unified",
    title_text="Global oil benchmark vs carrier fuel spend",
    legend=dict(orientation="v", yanchor="top", y=0.45, x=1.02, xanchor="left"),
)
show_plotly(fig_oil_fuel)

# Carrier-level averages: fare vs fuel % of revenue (full sample)
carr_avg = (
    fin.groupby("airline", as_index=False)
    .agg(
        passengers_carried_m=("passengers_carried_m", "mean"),
        fuel_cost_pct_revenue=("fuel_cost_pct_revenue", "mean"),
        profit_margin_pct=("profit_margin_pct", "mean"),
    )
    .merge(
        tick.groupby("airline", as_index=False).agg(
            mean_fare_usd=("total_fare_usd", "mean"),
            mean_load_factor=("load_factor_pct", "mean"),
        ),
        on="airline",
        how="inner",
    )
)
fig_sc = px.scatter(
    carr_avg,
    x="fuel_cost_pct_revenue",
    y="mean_fare_usd",
    size="passengers_carried_m",
    hover_name="airline",
    color="profit_margin_pct",
    color_continuous_scale=[[0, "#e8e8e8"], [1, "#1a1a1a"]],
    title="Carriers: mean fuel % of revenue vs mean fare (bubble ~ avg monthly passengers)",
)
fig_sc.update_layout(height=520, coloraxis_colorbar=dict(title="Avg profit margin %"))
show_plotly(fig_sc)


## Ticket prices, load factor, and fuel share of OpEx

Global monthly averages from ticket rows (all airlines combined).


In [6]:
tick_m = (
    tick.groupby("month_dt", as_index=False)
    .agg(
        mean_total_fare_usd=("total_fare_usd", "mean"),
        mean_load_factor_pct=("load_factor_pct", "mean"),
        mean_fuel_opex_pct=("fuel_cost_pct_opex", "mean"),
        mean_jet_fuel_usd_barrel=("jet_fuel_usd_barrel", "mean"),
    )
    .sort_values("month_dt")
)
if tick_m["mean_fuel_opex_pct"].max() <= 1.5:
    tick_m["fuel_opex_display"] = tick_m["mean_fuel_opex_pct"] * 100
else:
    tick_m["fuel_opex_display"] = tick_m["mean_fuel_opex_pct"]

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    specs=[[{"secondary_y": True}], [{"secondary_y": False}]],
    subplot_titles=(
        "Mean total fare (USD) vs jet fuel (right axis, USD/bbl)",
        "Mean load factor (%) and fuel share of OpEx (%)",
    ),
)
fig.add_trace(
    go.Scatter(
        x=tick_m["month_dt"],
        y=tick_m["mean_total_fare_usd"],
        name="Mean total fare",
        line=dict(color=PRISM[0], width=2),
    ),
    row=1,
    col=1,
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=tick_m["month_dt"],
        y=tick_m["mean_jet_fuel_usd_barrel"],
        name="Jet fuel USD/bbl",
        line=dict(color=PRISM[1], width=2),
    ),
    row=1,
    col=1,
    secondary_y=True,
)
fig.add_trace(
    go.Scatter(
        x=tick_m["month_dt"],
        y=tick_m["mean_load_factor_pct"],
        name="Load factor %",
        line=dict(color=PRISM[2], width=2),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=tick_m["month_dt"],
        y=tick_m["fuel_opex_display"],
        name="Fuel % of OpEx",
        line=dict(color=PRISM[3], width=2),
    ),
    row=2,
    col=1,
)
fig.update_yaxes(title_text="Fare (USD)", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Jet fuel (USD/bbl)", row=1, col=1, secondary_y=True)
for _label, (x0, x1, c) in PHASE_VRECT.items():
    for row in (1, 2):
        fig.add_vrect(
            x0=x0, x1=x1, fillcolor=c, opacity=0.1, layer="below", line_width=0, row=row, col=1
        )
fig.update_layout(
    height=720,
    title_text="Ticket-level aggregates by month",
    hovermode="x unified",
)
show_plotly(fig)


## Fuel surcharges and route cost impact

Mean surcharge by month (all bands). Top routes by `extra_fuel_cost_usd` (single snapshot per row).


In [7]:
sc_m = sc.groupby("month_dt", as_index=False)["fuel_surcharge_usd"].mean().sort_values("month_dt")
fig = go.Figure(
    go.Scatter(
        x=sc_m["month_dt"],
        y=sc_m["fuel_surcharge_usd"],
        name="Mean surcharge USD",
        mode="lines",
        line=dict(color=PRISM[5], width=2),
    )
)
for _label, (x0, x1, c) in PHASE_VRECT.items():
    fig.add_vrect(x0=x0, x1=x1, fillcolor=c, opacity=0.12, layer="below", line_width=0)
fig.update_layout(title="Mean fuel surcharge (USD) by month", height=400)
show_plotly(fig)

top_rt = rt.nlargest(25, "extra_fuel_cost_usd")[
    ["origin_city", "destination_city", "airline", "extra_fuel_cost_usd", "month"]
]
top_rt["route"] = top_rt["origin_city"] + " → " + top_rt["destination_city"]
fig2 = go.Figure(
    go.Bar(
        y=top_rt["route"][::-1],
        x=top_rt["extra_fuel_cost_usd"][::-1],
        orientation="h",
        marker_color=PRISM[6],
    )
)
fig2.update_layout(
    title="Top 25 routes by extra fuel cost (USD)",
    height=700,
)
show_plotly(fig2)


## Conflict oil events (sample)

Event table drives narrative context; use with caution for inference.


In [8]:
display(ev.head(12))

,event_date,event_type,event_description,location,severity,brent_before_usd,brent_after_usd,oil_price_change_pct,airfare_impact_pct,days_since_prev_event,conflict_phase,flight_cancellations_est,airspace_closures_countries,data_source,event_dt
0,2019-04-11,Political,Venezuela sanctions escalate; OPEC+ extends cuts,Global,Medium,67.5,72.3,7.1,2.5,NaN,Pre-Pandemic Baseline,1500,1,Reuters / Bloomberg / Al Jazeera / IATA / EIA ...,2019-04-11
1,2019-06-13,Military,Tanker attacks in Gulf of Oman — Iran blamed,Strait of Hormuz,High,60.5,65.2,7.8,3.1,63.0,Pre-Pandemic Baseline,1671,2,Reuters / Bloomberg / Al Jazeera / IATA / EIA ...,2019-06-13
2,2019-09-14,Military,Houthi drone strike hits Saudi Aramco faciliti...,"Abqaiq, Saudi Arabia",Very High,60.2,71.5,18.7,6.2,93.0,Pre-Pandemic Baseline,2737,3,Reuters / Bloomberg / Al Jazeera / IATA / EIA ...,2019-09-14
3,2019-12-27,Political,US kills Iranian General Soleimani — tensions ...,"Baghdad, Iraq",Very High,66.0,70.7,7.1,3.4,104.0,Pre-Pandemic Baseline,1801,1,Reuters / Bloomberg / Al Jazeera / IATA / EIA ...,2019-12-27
4,2020-01-08,Military,Iran fires missiles at US bases in Iraq,Iraq,Extreme,70.7,65.2,-7.7,4.8,12.0,Pre-Pandemic Baseline,2422,1,Reuters / Bloomberg / Al Jazeera / IATA / EIA ...,2020-01-08
5,2020-03-06,Economic,OPEC+ collapse — Saudi Arabia launches price war,"Vienna, Austria",Very High,51.9,31.7,-39.0,-12.5,58.0,COVID-19 Collapse,7870,8,Reuters / Bloomberg / Al Jazeera / IATA / EIA ...,2020-03-06
6,2020-04-20,Economic,WTI crude crashes to -$37/barrel amid COVID-19,Global / USA,Extreme,26.6,16.3,-38.7,-29.5,45.0,COVID-19 Collapse,15237,9,Reuters / Bloomberg / Al Jazeera / IATA / EIA ...,2020-04-20
7,2021-03-23,Operational,Suez Canal blocked by Ever Given (6 days),"Suez Canal, Egypt",High,63.8,66.3,3.9,1.8,337.0,COVID-19 Collapse,645,0,Reuters / Bloomberg / Al Jazeera / IATA / EIA ...,2021-03-23
8,2021-10-04,Political,OPEC+ rejects calls to boost production — supp...,Vienna,High,79.3,84.1,6.1,3.4,195.0,Recovery & Surge,1889,1,Reuters / Bloomberg / Al Jazeera / IATA / EIA ...,2021-10-04
9,2022-02-24,Military,Russia invades Ukraine — massive oil price spike,Ukraine,Extreme,97.5,128.0,31.3,14.2,143.0,Recovery & Surge,5185,6,Reuters / Bloomberg / Al Jazeera / IATA / EIA ...,2022-02-24


## Correlation (monthly merged panel)

Merge industry financials with ticket averages and oil series on `month_dt`. Heatmap color encodes |ρ| (white = 0, dark = 1); each cell text shows signed *r*.


In [9]:
panel = fin_m.merge(tick_m, on="month_dt", how="inner").merge(
    oil_sorted[["month_dt", "brent_crude_usd_barrel", "jet_fuel_usd_barrel"]],
    on="month_dt",
    how="inner",
)
num_cols = [
    "revenue_usd_m",
    "fuel_cost_usd_m",
    "net_profit_usd_m",
    "passengers_carried_m",
    "mean_total_fare_usd",
    "mean_load_factor_pct",
    "mean_jet_fuel_usd_barrel",
    "fuel_opex_display",
    "brent_crude_usd_barrel",
    "jet_fuel_usd_barrel",
]

corr = panel[num_cols].corr()
# Single-hue scale: |ρ| as shading (cells still annotated with signed r)
z_abs = np.abs(corr.values)
fig = go.Figure(
    data=go.Heatmap(
        z=z_abs,
        x=corr.columns,
        y=corr.columns,
        zmin=0,
        zmax=1,
        colorscale=[[0, "rgb(255,255,255)"], [1, "rgb(35,35,35)"]],
        text=np.round(corr.values, 2),
        texttemplate="%{text}",
        colorbar=dict(title="|r|"),
    )
)
fig.update_layout(title="Correlation matrix (monthly panel)", height=700, width=800)
show_plotly(fig)


## Radar comparison (normalized profiles)

Each axis is **min–max scaled to 0–100** within the compared set (higher = more relative to peers on that metric). Carriers use airline-level aggregates; **region** follows the dataset’s `region` field; **conflict phase** aggregates all carrier-month rows in that phase. Use for pattern comparison, not levels across groups.


In [ ]:
def _norm_cols(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        lo, hi = out[c].min(), out[c].max()
        out[c + "_n"] = 50.0 if hi == lo else 100.0 * (out[c] - lo) / (hi - lo)
    return out


def add_radar_traces(
    fig,
    sub: pd.DataFrame,
    name_col: str,
    rcols: list[str],
    labels: list[str],
    *,
    fill: str = "toself",
    line_width: float = 2,
    opacity: float = 0.2,
):
    base_cols = [c.replace("_n", "") for c in rcols]
    sub = sub.dropna(subset=base_cols)
    for _, row in sub.iterrows():
        rvals = [float(row[c]) for c in rcols] + [float(row[rcols[0]])]
        theta = labels + [labels[0]]
        fig.add_trace(
            go.Scatterpolar(
                r=rvals,
                theta=theta,
                fill=fill,
                name=str(row[name_col]),
                opacity=opacity if fill == "toself" else 1.0,
                line=dict(width=line_width),
            )
        )


# --- Carriers (top 6 by passengers) ---
fin_c = fin.groupby("airline", as_index=False).agg(
    passengers_carried_m=("passengers_carried_m", "sum"),
    revenue_usd_m=("revenue_usd_m", "sum"),
    fuel_cost_pct_revenue=("fuel_cost_pct_revenue", "mean"),
    profit_margin_pct=("profit_margin_pct", "mean"),
    net_profit_usd_m=("net_profit_usd_m", "sum"),
)
fin_c = fin_c.merge(
    tick.groupby("airline", as_index=False).agg(
        mean_fare_usd=("total_fare_usd", "mean"),
        mean_load_factor=("load_factor_pct", "mean"),
    ),
    on="airline",
    how="inner",
)
m_carrier = [
    ("passengers_carried_m", "Passengers (sum)"),
    ("revenue_usd_m", "Revenue (sum)"),
    ("mean_fare_usd", "Mean fare"),
    ("mean_load_factor", "Load factor"),
    ("fuel_cost_pct_revenue", "Fuel % rev"),
    ("profit_margin_pct", "Profit margin %"),
]
cols_c = [m[0] for m in m_carrier]
lab_c = [m[1] for m in m_carrier]
top6 = fin_c.nlargest(6, "passengers_carried_m")
top6n = _norm_cols(top6, cols_c)
fig_rc = go.Figure()
add_radar_traces(
    fig_rc,
    top6n,
    "airline",
    [c + "_n" for c in cols_c],
    lab_c,
    fill="toself",
    opacity=0.22,
)
fig_rc.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100], tickvals=[0, 25, 50, 75, 100])),
    title="Carriers — top 6 by total passengers (normalized)",
    height=560,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=-0.2),
)
show_plotly(fig_rc)

# --- Region (ticket file `region`; macro geography in this dataset) ---
fin_r = fin.groupby("region", as_index=False).agg(
    passengers_carried_m=("passengers_carried_m", "sum"),
    revenue_usd_m=("revenue_usd_m", "sum"),
    fuel_cost_pct_revenue=("fuel_cost_pct_revenue", "mean"),
    profit_margin_pct=("profit_margin_pct", "mean"),
)
tr = tick.groupby("region", as_index=False).agg(
    mean_fare_usd=("total_fare_usd", "mean"),
    mean_load_factor=("load_factor_pct", "mean"),
)
fin_r = fin_r.merge(tr, on="region", how="inner")
m_reg = [
    ("passengers_carried_m", "Passengers"),
    ("revenue_usd_m", "Revenue"),
    ("mean_fare_usd", "Mean fare"),
    ("mean_load_factor", "Load factor"),
    ("fuel_cost_pct_revenue", "Fuel % rev"),
    ("profit_margin_pct", "Profit margin %"),
]
cols_r = [m[0] for m in m_reg]
lab_r = [m[1] for m in m_reg]
fin_rn = _norm_cols(fin_r, cols_r)
fig_rr = go.Figure()
add_radar_traces(
    fig_rr,
    fin_rn,
    "region",
    [c + "_n" for c in cols_r],
    lab_r,
    fill="toself",
    opacity=0.25,
)
fig_rr.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title="Regions — aggregated carriers (normalized)",
    height=560,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=-0.25),
)
show_plotly(fig_rr)

# --- Conflict phase (time / narrative periods) ---
fin_p = fin.groupby("conflict_phase", as_index=False).agg(
    passengers_carried_m=("passengers_carried_m", "sum"),
    revenue_usd_m=("revenue_usd_m", "sum"),
    fuel_cost_pct_revenue=("fuel_cost_pct_revenue", "mean"),
    profit_margin_pct=("profit_margin_pct", "mean"),
)
tick_p = tick.groupby("conflict_phase", as_index=False).agg(
    mean_fare_usd=("total_fare_usd", "mean"),
    mean_load_factor=("load_factor_pct", "mean"),
)
fin_p = fin_p.merge(tick_p, on="conflict_phase", how="inner")
fin_p["conflict_phase"] = pd.Categorical(
    fin_p["conflict_phase"], categories=PHASE_ORDER, ordered=True
)
fin_p = fin_p.sort_values("conflict_phase")
m_ph = [
    ("passengers_carried_m", "Passengers"),
    ("revenue_usd_m", "Revenue"),
    ("mean_fare_usd", "Mean fare"),
    ("mean_load_factor", "Load factor"),
    ("fuel_cost_pct_revenue", "Fuel % rev"),
    ("profit_margin_pct", "Profit margin %"),
]
cols_p = [m[0] for m in m_ph]
lab_p = [m[1] for m in m_ph]
fin_pn = _norm_cols(fin_p, cols_p)
fig_rp = go.Figure()
add_radar_traces(
    fig_rp,
    fin_pn,
    "conflict_phase",
    [c + "_n" for c in cols_p],
    lab_p,
    fill="none",
    line_width=2.2,
)
fig_rp.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title="Conflict / narrative phase — industry aggregates (normalized, lines only)",
    height=620,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=-0.35, font=dict(size=10)),
)
show_plotly(fig_rp)

# --- Country (top 8 by passengers) ---
fin_ct = fin.groupby("country", as_index=False).agg(
    passengers_carried_m=("passengers_carried_m", "sum"),
    revenue_usd_m=("revenue_usd_m", "sum"),
    fuel_cost_pct_revenue=("fuel_cost_pct_revenue", "mean"),
    profit_margin_pct=("profit_margin_pct", "mean"),
)
fin_ct = fin_ct.merge(
    tick.groupby("country", as_index=False).agg(
        mean_fare_usd=("total_fare_usd", "mean"),
        mean_load_factor=("load_factor_pct", "mean"),
    ),
    on="country",
    how="inner",
)
top_ct = fin_ct.nlargest(8, "passengers_carried_m")
top_ctn = _norm_cols(top_ct, cols_c)
fig_rct = go.Figure()
add_radar_traces(
    fig_rct,
    top_ctn,
    "country",
    [c + "_n" for c in cols_c],
    lab_c,
    fill="none",
    line_width=2,
)
fig_rct.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title="Countries — top 8 by total passengers (normalized, lines only)",
    height=560,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=-0.22),
)
show_plotly(fig_rct)


## Regression

- **Bivariate (visual)**: scatter of fare vs jet fuel with OLS line.
- **Multivariate**: OLS for mean total fare on jet fuel, load factor, and conflict phase (categorical).


In [10]:
sample = tick.dropna(
    subset=["total_fare_usd", "jet_fuel_usd_barrel", "load_factor_pct", "conflict_phase"]
).copy()
fig = px.scatter(
    sample.sample(min(8000, len(sample)), random_state=42),
    x="jet_fuel_usd_barrel",
    y="total_fare_usd",
    color="conflict_phase",
    color_discrete_map=PHASE_COLORS,
    trendline="ols",
    title="Total fare vs jet fuel (sample up to 8k rows, OLS trendline)",
    opacity=0.35,
)
fig.update_layout(height=600)
show_plotly(fig)

import statsmodels.formula.api as smf

reg = (
    tick.dropna(subset=["total_fare_usd", "jet_fuel_usd_barrel", "load_factor_pct", "conflict_phase"])
    .sample(min(12000, len(tick)), random_state=43)
)
model = smf.ols(
    "total_fare_usd ~ jet_fuel_usd_barrel + load_factor_pct + C(conflict_phase)",
    data=reg,
).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:         total_fare_usd   R-squared:                       0.104
Model:                            OLS   Adj. R-squared:                  0.103
Method:                 Least Squares   F-statistic:                     153.9
Date:                Sun, 22 Mar 2026   Prob (F-statistic):          1.09e-276
Time:                        15:10:37   Log-Likelihood:            -1.0385e+05
No. Observations:               12000   AIC:                         2.077e+05
Df Residuals:                   11990   BIC:                         2.078e+05
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------

## Limitations

Observational, compiled/synthetic-style dataset; phase labels and event windows are narrative aids. Correlation and OLS do not establish causal effects of conflicts or oil on fares or profits. Hedge positions (`fuel_hedging_pct`) and accounting differ by carrier.
